# Vaidya — Phase 2: QLoRA Training (Colab T4 ×2)

**Before running:**
1. Runtime → Change runtime type → **GPU T4 ×2**
2. Left sidebar → 🔑 Secrets → add:
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `AWS_DEFAULT_REGION` → `ap-south-1`
3. Toggle **Notebook access ON** for each secret

**Expected runtime:** ~5–7 hours (T4 ×2 is faster than P100)

## Step 1 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Change runtime type to GPU T4 ×2.'

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB  CC {p.major}.{p.minor}')

print(f'BF16 supported : {torch.cuda.is_bf16_supported()}')   # False on T4
print(f'Total GPUs     : {torch.cuda.device_count()}')        # 2 on T4×2

## Step 2 — Install packages

In [ ]:
%%capture
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.11.0 \
    bitsandbytes==0.43.3 \
    accelerate==0.34.0 \
    datasets==3.0.0 \
    mlflow==2.15.0 \
    boto3 awscli
print('Install done.')

## Step 3 — Upload training scripts

In [ ]:
# Option A: clone from GitHub (fastest once you push the repo)
# !git clone https://github.com/Anand09-in/vaidya.git
# import sys; sys.path.insert(0, './vaidya/layer2_finetune')

# Option B: upload train.py + qlora_config.py via Files panel (left sidebar)
# then verify:
import os
for f in ['train.py', 'qlora_config.py']:
    print('✅' if os.path.exists(f) else '❌ MISSING — upload via Files panel', f)

## Step 4 — AWS credentials + pull data from S3

In [ ]:
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = userdata.get('AWS_DEFAULT_REGION')

# Quick connectivity test
!aws s3 ls s3://vaidya-artifacts/ 2>&1 | head -5

In [ ]:
os.makedirs('./data', exist_ok=True)
!aws s3 sync s3://vaidya-artifacts/data/ ./data/ --include '*.parquet'

import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'./data/{split}.parquet')
    print(f'{split}: {len(df):,} rows')

## Step 5 — Run training

`device_map='auto'` in `train.py` automatically spreads Mistral-7B across both T4 GPUs.
~15 GB per GPU → comfortable with batch_size=2, seq_len=640.

In [ ]:
# Confirm both GPUs are visible to the training script
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ⚡ FULL TRAINING RUN — ~5–7 hours
# If OOM: add --batch-size 1 --grad-accum 32
!python train.py --run-name run1

## Step 6 — Verify checkpoint on S3

In [ ]:
!aws s3 ls s3://vaidya-artifacts/checkpoints/run1/ --recursive --human-readable
print()
!aws s3 ls s3://vaidya-artifacts/mlflow/

## Step 7 — Loss sanity check

In [ ]:
import json, glob

states = glob.glob('./checkpoints/run1/**/trainer_state.json', recursive=True)
if states:
    with open(sorted(states)[-1]) as f:
        state = json.load(f)
    history = state.get('log_history', [])
    train_l = [(e['step'], e['loss'])      for e in history if 'loss' in e]
    eval_l  = [(e['step'], e['eval_loss']) for e in history if 'eval_loss' in e]
    if train_l:
        print(f'Train loss — step {train_l[0][0]}: {train_l[0][1]:.4f}  →  step {train_l[-1][0]}: {train_l[-1][1]:.4f}')
    if eval_l:
        print(f'Eval  loss — step {eval_l[0][0]}: {eval_l[0][1]:.4f}  →  step {eval_l[-1][0]}: {eval_l[-1][1]:.4f}')
else:
    print('trainer_state.json not found — training may still be running.')

## Gate check before Phase 3

- ✅ Training loss decreased steadily
- ✅ Eval loss decreased and is not diverging
- ✅ `s3://vaidya-artifacts/checkpoints/run1/` has adapter files
- ✅ `s3://vaidya-artifacts/mlflow/mlflow.db` exists